In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

In [ ]:
# Define paths
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")
networks_path = base_path / "Processed_data/networks"
damages_path = base_path / "Processed_data/direct_damages_summary_uids"
hydrobasins_path = base_path / "Processed_data/HydroBASINS_Level12_Clipped_Jamaica.shp"


In [ ]:
# Define subfolders for networks
subfolders = ["energy", "transport", "buildings", "water"]

In [ ]:
# Define Jamaica CRS
jamaica_metric_grid_crs = "EPSG:3448"

In [ ]:
# Load HydroBASINS data
hydrobasins = gpd.read_file(hydrobasins_path)
hydrobasins = hydrobasins.to_crs(jamaica_metric_grid_crs)
print(f"Loaded Hydrobasins with {len(hydrobasins)} polygons.")

In [ ]:
# Load network data
network_data = {}

for subfolder in subfolders:
    folder_path = networks_path / subfolder
    gpkg_files = folder_path.glob("*.gpkg")
    
    for gpkg_file in gpkg_files:
        try:
            data = gpd.read_file(gpkg_file)
            if data.crs != jamaica_metric_grid_crs:
                data = data.to_crs(jamaica_metric_grid_crs)
            network_data[gpkg_file.stem] = data
            print(f"Loaded and reprojected: {gpkg_file.name}")
        except Exception as e:
            print(f"Failed to load {gpkg_file.name}: {e}")

print(f"Successfully loaded {len(network_data)} network layers.")


In [ ]:
# Load damages data (EAEL only)
damages_data = {}

parquet_files = damages_path.glob("*.parquet")

for file in parquet_files:
    if "EAEL" in file.stem:  # Only process files with "EAEL" in the name
        data = pd.read_parquet(file)
        if 'hazard' in data.columns:
            fluvial_data = data[data['hazard'] == 'fluvial']
            if len(fluvial_data) > 0:
                damages_data[file.stem.split("_")[0]] = fluvial_data
                print(f"Loaded and filtered (EAEL): {file.name} with {len(fluvial_data)} fluvial rows.")
            else:
                print(f"No fluvial rows in: {file.name}")
        else:
            print(f"No 'hazard' column in: {file.name}")

print(f"Successfully loaded {len(damages_data)} damages datasets.")


In [ ]:
# Match damages to network layers
joined_networks = {}

for network_name, network_gdf in network_data.items():
    for damages_name, damages_df in damages_data.items():
        # Identify the correct join key
        join_key = None
        for col in ['edge_id', 'node_id', 'osm_id', 'id', 'uid']:
            if col in network_gdf.columns and col in damages_df.columns:
                join_key = col
                break

        if join_key:
            # Perform the join
            print(f"Joining {network_name} and {damages_name} on key '{join_key}'")
            joined = network_gdf.merge(damages_df, on=join_key, how="inner")
            joined_networks[f"{network_name}_{damages_name}"] = joined
            print(f"Joined {network_name} with {damages_name}. Result: {len(joined)} rows.")
        else:
            print(f"No common key found for {network_name} and {damages_name}. Skipping...")

print(f"Successfully joined {len(joined_networks)} network-damages combinations.")
